In [3]:

from src.data_preprocessing import load_and_prepare_data
from src.model import get_trained_classical_models, build_dl_model

X_train, X_test, y_train, y_test, scaler = load_and_prepare_data("balanced_heart_dataset.csv")

log_model, svm_model, rf_model, xgb_model = get_trained_classical_models(X_train, y_train)

dl_model = build_dl_model(X_train.shape[1])
%load_ext autoreload
%autoreload 2

from src.data_preprocessing import load_and_prepare_data
from src.model import build_dl_model, get_trained_classical_models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

csv_path = r"C:\Users\osvis\OneDrive\Desktop\Health and Lifestyle Analysis\balanced_heart_dataset.csv"
X_train_scaled, X_test_scaled, y_train, y_test, scaler = load_and_prepare_data(csv_path)

dl_model = build_dl_model(X_train_scaled.shape[1])

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=6,
    restore_best_weights=True
)

history = dl_model.fit(
    X_train_scaled,
    y_train,
    epochs=50,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

loss, accuracy = dl_model.evaluate(X_test_scaled, y_test)
print("\nTest Accuracy:", round(accuracy*100, 2), "%")

y_pred_prob = dl_model.predict(X_test_scaled)
y_pred = (y_pred_prob > 0.5).astype(int)

print("\n--- Performance Metrics ---")
print("Precision:", round(precision_score(y_test, y_pred), 3))
print("Recall:", round(recall_score(y_test, y_pred), 3))
print("F1 Score:", round(f1_score(y_test, y_pred), 3))

print("\nConfusion Matrix\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report\n", classification_report(y_test, y_pred))
dl_model.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.2, verbose=0)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Epoch 1/50
174/174 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7789 - loss: 0.4542 - val_accuracy: 0.8522 - val_loss: 0.3648
Epoch 2/50
174/174 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8326 - loss: 0.3637 - val_accuracy: 0.8629 - val_loss: 0.3142
Epoch 3/50
174/174 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8430 - loss: 0.3471 - val_accuracy: 0.8676 - val_loss: 0.2995
Epoch 4/50
174/174 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8470 - loss: 0.3406 - val_accuracy: 0.8694 - val_loss: 0.2966
Epoch 5/50
174/174 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8510 - loss: 0.3297 - val_accuracy: 0.8712 - val_loss: 0.2899
Epoch 6/50
174/174 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8550 - loss: 0.3221 - val_accuracy: 0.8755 - val_loss: 0.2832
Epoch 7/50
174/174 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8614 - loss: 0.3130 - val_accuracy: 0.8766 - val_loss: 0.2799
Epoch 8/50
174/1

In [6]:
from src.visualization import plot_model_comparison

all_models = (log_model, svm_model, rf_model, xgb_model, dl_model)
plot_model_comparison(all_models, X_test_scaled, y_test)

ImportError: cannot import name 'get_performance_dataframe' from 'src.model' (c:\Users\osvis\OneDrive\Desktop\Health and Lifestyle Analysis\src\model.py)

In [5]:
from src.recommendations import get_user_input, generate_recommendations

feature_columns = ['Age', 'Gender', 'Height_cm', 'Weight_kg', 'BMI', 'Systolic_BP',
                   'Diastolic_BP', 'Cholesterol_Total', 'Cholesterol_LDL',
                   'Cholesterol_HDL', 'Fasting_Blood_Sugar', 'Smoking_Status',
                   'Alcohol_Consumption', 'Physical_Activity_Level', 'Family_History',
                   'Stress_Level', 'Sleep_Hours']

user_df = get_user_input(feature_columns)

user_scaled = scaler.transform(user_df)

pred_lr = log_model.predict(user_scaled)[0]
pred_svm = svm_model.predict(user_scaled)[0]
pred_rf = rf_model.predict(user_scaled)[0]
pred_xgb = xgb_model.predict(user_scaled)[0]
pred_dl = (dl_model.predict(user_scaled) > 0.5).astype(int)[0][0]

votes = [pred_lr, pred_svm, pred_rf, pred_xgb, pred_dl]
vote_count = sum(votes)
final_risk = 1 if vote_count >= 3 else 0

print("\n====================================")
print("       PATIENT DIAGNOSTIC REPORT     ")
print("====================================")
print(f"Final Prediction: {'HIGH RISK ' if final_risk == 1 else 'LOW RISK ✅'}")
print(f"Ensemble Confidence: {vote_count / len(votes) * 100:.1f}%")

print("\n--- Tailored Preventative Guidelines ---")
recs = generate_recommendations(user_df)
for r in recs:
    print(r)
print("====================================")


--- Patient Medical Input Console ---


ValueError: could not convert string to float: ''